# C++23 FX Option Pricer — Jupyter Demo

This notebook demonstrates the **pybind11 Python wrapper** around the C++ pricing engines.
All computation runs at native C++ speed; Python handles visualization and analysis.

**Instructions:** Run Cell 1 first (~2 min), then run the remaining cells.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1: Setup — install GCC 14, build module, import
# ══════════════════════════════════════════════════════════════
import subprocess, sys, os, glob

# 1. Install GCC 14 for C++23 support
print('Installing GCC 14...')
subprocess.run('sudo add-apt-repository -y ppa:ubuntu-toolchain-r/test 2>/dev/null', shell=True, capture_output=True)
subprocess.run('sudo apt update -qq 2>/dev/null', shell=True, capture_output=True)
subprocess.run('sudo apt install -y gcc-14 g++-14 2>/dev/null', shell=True, capture_output=True)
subprocess.run('pip install -q pybind11', shell=True, capture_output=True)
print('GCC 14 installed')

# 2. Force-load GCC 14's libstdc++ (Colab ships an older version)
import ctypes
libs = sorted(glob.glob('/usr/lib/x86_64-linux-gnu/libstdc++.so.6.0.*'))
if libs:
    ctypes.CDLL(libs[-1], mode=ctypes.RTLD_GLOBAL)
    print(f'Loaded {libs[-1]}')

# 3. Clone repo
if not os.path.exists('/content/cpp20-option-pricer/CMakeLists.txt'):
    subprocess.run('git clone https://github.com/quynhnguyen141299-hub/cpp20-option-pricer.git /content/cpp20-option-pricer', shell=True, capture_output=True)
    print('Repo cloned')
else:
    subprocess.run('cd /content/cpp20-option-pricer && git pull', shell=True, capture_output=True)
    print('Repo updated')

# 4. Build Python module
so_files = glob.glob('/content/cpp20-option-pricer/build/fx_pricer*.so')
if not so_files:
    print('Building C++ module...')
    pybind11_dir = subprocess.run(
        'python3 -c "import pybind11; print(pybind11.get_cmake_dir())"',
        shell=True, capture_output=True, text=True).stdout.strip()
    r = subprocess.run(
        f'cd /content/cpp20-option-pricer && rm -rf build && '
        f'cmake -DCMAKE_CXX_COMPILER=g++-14 -DBUILD_PYTHON=ON '
        f'-Dpybind11_DIR={pybind11_dir} -B build 2>&1 && '
        f'cmake --build build 2>&1',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('BUILD FAILED:'); print(r.stdout[-2000:]); print(r.stderr[-2000:])
    else:
        print('Build complete')
else:
    print('Module already built')

# 5. Import
sys.path.insert(0, '/content/cpp20-option-pricer/build')
import fx_pricer as fp
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print('\nReady:', [x for x in dir(fp) if not x.startswith('_')])

---
## 1. BS vs MC Cross-Validation

Price a EURUSD call with both engines and verify they agree within Monte Carlo standard error.

In [ ]:
mkt = fp.MarketSnap(S=1.085, sigma=0.075, r_d=0.0435, r_f=0.025)
call = fp.Contract(K=1.09, T=0.25, type=fp.OptType.Call)

bs = fp.BSEngine()
bs_r = bs.price(call, mkt)
print(f"BS price:  {bs_r.price:.8f}")
print(f"Greeks:    Δ={bs_r.greeks.delta:+.6f}  Γ={bs_r.greeks.gamma:.6f}  "
      f"V={bs_r.greeks.vega:.6f}  Θ={bs_r.greeks.theta:.6f}")

cfg = fp.MCConfig()
cfg.n_paths = 500_000
cfg.antithetic = True
cfg.control_variate = True
cfg.n_threads = 1
mc = fp.MCEngine(r_d=0.0435, sigma=0.075, config=cfg)
mc_r = mc.price(call, mkt)
print(f"\nMC price:  {mc_r.price:.8f} ± {mc_r.std_err:.2e}")
print(f"Diff:      {abs(bs_r.price - mc_r.price):.2e} (< 1 SE = {mc_r.std_err:.2e})")

---
## 2. Greeks Surface — Delta & Gamma vs Spot

Sweep spot from 1.05 to 1.13 and plot how delta and gamma evolve.

In [ ]:
spots = np.linspace(1.05, 1.13, 80)
deltas, gammas = [], []

for s in spots:
    m = fp.MarketSnap(S=s, sigma=0.075, r_d=0.0435, r_f=0.025)
    r = bs.price(call, m)
    deltas.append(r.greeks.delta)
    gammas.append(r.greeks.gamma)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(spots, deltas, 'b-', linewidth=2)
ax1.axvline(1.09, color='grey', linestyle='--', alpha=0.5, label='Strike')
ax1.set_xlabel('Spot')
ax1.set_ylabel('Delta')
ax1.set_title('Call Delta vs Spot')
ax1.legend()

ax2.plot(spots, gammas, 'r-', linewidth=2)
ax2.axvline(1.09, color='grey', linestyle='--', alpha=0.5, label='Strike')
ax2.set_xlabel('Spot')
ax2.set_ylabel('Gamma')
ax2.set_title('Call Gamma vs Spot')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 3. Heston Volatility Smile

The Heston model generates a realistic implied vol smile. Negative ρ → downside skew.

In [ ]:
params = fp.HestonParams()
params.v0 = 0.04
params.kappa = 1.5
params.theta = 0.04
params.xi = 0.3

S, r_d, r_f, T = 100.0, 0.05, 0.0, 0.5
moneyness = np.linspace(0.80, 1.20, 15)
strikes = S * moneyness

fig, ax = plt.subplots(figsize=(10, 6))

for rho, color, label in [(-0.7, '#2563eb', 'ρ = −0.7 (equity-like skew)'),
                           (0.0,  '#6b7280', 'ρ =  0.0 (symmetric smile)'),
                           (0.5,  '#dc2626', 'ρ = +0.5 (inverse skew)')]:
    params.rho = rho
    cfg = fp.MCConfig()
    cfg.n_paths = 200_000
    cfg.steps = 50
    cfg.antithetic = True
    cfg.n_threads = 1
    hmc = fp.HestonMCEngine(r_d=r_d, params=params, config=cfg)
    mkt_h = fp.MarketSnap(S=S, sigma=0.20, r_d=r_d, r_f=r_f)

    ivs = []
    for K in strikes:
        c = fp.Contract(K=K, T=T, type=fp.OptType.Call)
        r = hmc.price(c, mkt_h)
        try:
            iv = bs.implied_vol(c, mkt_h, r.price)
            ivs.append(iv * 100)
        except:
            ivs.append(np.nan)

    ax.plot(moneyness, ivs, 'o-', color=color, linewidth=2, markersize=4, label=label)

ax.set_xlabel('Moneyness (K / S)')
ax.set_ylabel('Implied Volatility (%)')
ax.set_title('Heston Implied Volatility Smile — Effect of ρ')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Barrier Option — Knockout Probability

Down-and-Out call: as the barrier approaches spot, the option loses value rapidly.

In [ ]:
mkt_b = fp.MarketSnap(S=100.0, sigma=0.20, r_d=0.05, r_f=0.0)

vanilla = bs.price(fp.Contract(K=100.0, T=1.0, type=fp.OptType.Call), mkt_b)

cfg_b = fp.MCConfig()
cfg_b.n_paths = 200_000
cfg_b.steps = 100
cfg_b.antithetic = True
cfg_b.n_threads = 1
bmc = fp.BarrierMCEngine(r_d=0.05, sigma=0.20, config=cfg_b)

barriers = np.linspace(60, 99, 25)
pct_vanilla = []

for B in barriers:
    bc = fp.BarrierContract(
        K=100.0, T=1.0, type=fp.OptType.Call,
        barrier=B, barrier_type=fp.BarrierType.DownAndOut
    )
    r = bmc.price_barrier(bc, mkt_b)
    pct_vanilla.append(100 * r.price / vanilla.price)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(barriers, pct_vanilla, 'o-', color='#2563eb', linewidth=2, markersize=4)
ax.axhline(100, color='grey', linestyle='--', alpha=0.5, label='Vanilla (100%)')
ax.fill_between(barriers, pct_vanilla, alpha=0.1, color='#2563eb')
ax.set_xlabel('Barrier Level')
ax.set_ylabel('% of Vanilla Price')
ax.set_title('Down-and-Out Call — Price vs Barrier Level (S = 100, K = 100)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. FD PDE Solver — American vs European Early Exercise

Sweep spot to show where the American put's early exercise premium is largest.

In [ ]:
fd = fp.FDEngine()
spots_fd = np.linspace(70, 130, 50)

eu_prices, am_prices, premiums = [], [], []

for s in spots_fd:
    m = fp.MarketSnap(S=s, sigma=0.20, r_d=0.05, r_f=0.0)
    eu_put = fp.Contract(K=100.0, T=1.0, type=fp.OptType.Put)
    am_put = fp.Contract(K=100.0, T=1.0, type=fp.OptType.Put, exercise=fp.Exercise.American)

    eu_r = fd.price(eu_put, m)
    am_r = fd.price(am_put, m)
    eu_prices.append(eu_r.price)
    am_prices.append(am_r.price)
    premiums.append(am_r.price - eu_r.price)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(spots_fd, eu_prices, 'b-', linewidth=2, label='European Put')
ax1.plot(spots_fd, am_prices, 'r--', linewidth=2, label='American Put')
ax1.plot(spots_fd, np.maximum(100 - spots_fd, 0), 'k:', linewidth=1, alpha=0.5, label='Intrinsic')
ax1.axvline(100, color='grey', linestyle='--', alpha=0.3)
ax1.set_xlabel('Spot')
ax1.set_ylabel('Price')
ax1.set_title('European vs American Put')
ax1.legend()

ax2.fill_between(spots_fd, premiums, alpha=0.3, color='#dc2626')
ax2.plot(spots_fd, premiums, 'r-', linewidth=2)
ax2.axvline(100, color='grey', linestyle='--', alpha=0.3)
ax2.set_xlabel('Spot')
ax2.set_ylabel('Premium')
ax2.set_title('Early Exercise Premium')

plt.tight_layout()
plt.show()

---
## 6. MC Convergence — Paths vs Price Error

Show how MC price converges to the BS analytical answer as we increase the number of paths.

In [ ]:
mkt_cv = fp.MarketSnap(S=1.085, sigma=0.075, r_d=0.0435, r_f=0.025)
call_cv = fp.Contract(K=1.09, T=0.25, type=fp.OptType.Call)
bs_ref = bs.price(call_cv, mkt_cv).price

path_counts = [1000, 2000, 5000, 10_000, 20_000, 50_000, 100_000, 200_000, 500_000]
errors_plain, errors_cv = [], []

for n in path_counts:
    cfg_p = fp.MCConfig()
    cfg_p.n_paths = n
    cfg_p.antithetic = False
    cfg_p.control_variate = False
    cfg_p.n_threads = 1
    mc_p = fp.MCEngine(r_d=0.0435, sigma=0.075, config=cfg_p)
    r_p = mc_p.price(call_cv, mkt_cv)
    errors_plain.append(abs(r_p.price - bs_ref))

    cfg_cv = fp.MCConfig()
    cfg_cv.n_paths = n
    cfg_cv.antithetic = True
    cfg_cv.control_variate = True
    cfg_cv.n_threads = 1
    mc_cv = fp.MCEngine(r_d=0.0435, sigma=0.075, config=cfg_cv)
    r_cv = mc_cv.price(call_cv, mkt_cv)
    errors_cv.append(abs(r_cv.price - bs_ref))

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(path_counts, errors_plain, 'o-', color='#dc2626', linewidth=2, label='Plain MC')
ax.loglog(path_counts, errors_cv, 's-', color='#2563eb', linewidth=2, label='Antithetic + Control Variate')

ref = errors_plain[0] * np.sqrt(path_counts[0]) / np.sqrt(path_counts)
ax.loglog(path_counts, ref, 'k--', alpha=0.3, label='O(1/√N) reference')

ax.set_xlabel('Number of Paths')
ax.set_ylabel('|MC − BS| Price Error')
ax.set_title('MC Convergence: Plain vs Variance-Reduced')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Implied Vol Solver Round-Trip

Price → IV → re-price: the Newton-Raphson solver recovers the original vol to machine precision.

In [ ]:
vols = np.linspace(0.03, 0.40, 20)
recovered, errors_iv = [], []

for v in vols:
    m = fp.MarketSnap(S=1.085, sigma=v, r_d=0.0435, r_f=0.025)
    c = fp.Contract(K=1.09, T=0.25, type=fp.OptType.Call)
    px = bs.price(c, m).price

    guess = fp.MarketSnap(S=1.085, sigma=0.30, r_d=0.0435, r_f=0.025)
    iv = bs.implied_vol(c, guess, px)
    recovered.append(iv * 100)
    errors_iv.append(abs(iv - v))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(vols * 100, recovered, 'b-', linewidth=2, label='Recovered IV')
ax1.plot(vols * 100, vols * 100, 'k--', alpha=0.3, label='Perfect recovery')
ax1.set_xlabel('True Vol (%)')
ax1.set_ylabel('Recovered IV (%)')
ax1.set_title('IV Solver Accuracy')
ax1.legend()

ax2.semilogy(vols * 100, errors_iv, 'r-', linewidth=2)
ax2.axhline(2.22e-16, color='grey', linestyle='--', alpha=0.5, label='Machine epsilon')
ax2.set_xlabel('True Vol (%)')
ax2.set_ylabel('|IV − True Vol|')
ax2.set_title('IV Solver Error (log scale)')
ax2.legend()

plt.tight_layout()
plt.show()

---

**Summary:** All pricing engines are accessible from Python via pybind11. The C++ library handles the computation; Python handles the visualization and analysis workflow — the standard quant desk pattern.